## NLP: Exploring Pretrained Transformers: BERT, GPT-2, T5

This notebook explores concepts from Géron’s Chapter 15. I explored pretrained Transformer models nu adjusting prompts and generation settings. 

* **Natural Language Processing (NLP)** enables computers to analyse and generate human language. It uses tokens, which may represent words, parts of words or punctuation.

* **Tokenisation:** A tokeniser splits text into tokens and converts them into numerical IDs the model can process.
* **Attention:** Allows the model to combine information from relevant tokens when representing or predicting a token.

* Earlier neural approaches used **recurrent neural networks (RNNs)** to process sequences step by step.
* Transformers use an attention mechanism to connect information across tokens and establish relationships over a longer context.

Through experiments with **BERT, GPT-2 and FLAN-T5**, I explored masked-token prediction, text generation and summarisation. The results showed that context and generation settings influence outputs, but confidence and fluency do not guarantee correctness. Confident outputs is not always plausible and can be wrong demonstrating why human review matters.

Different NLP tasks require different evaluation metrics:
* **BLEU** measures word-sequence overlap with reference text, used for translation.
* **ROUGE** measures overlap through shared words, sequences or subsequences, used for summarisation.
* **Perplexity** measures how well a language model predicts observed tokens, with lower values indicating better prediction.

## 1. BERT - predicting a word

In [ ]:
# Installing library and loading pretrained BERT
%pip install transformers

from transformers import pipeline

fill_mask = pipeline(
    "fill-mask",
    model="google-bert/bert-base-uncased",
    device=-1
)

- BERT's pipeline handles text preparation, model execution and output formatting
- "fill-mask" selects the missing-token prediction task.
- model=... loads pretrained BERT and its tokenizer.
- device=-1 runs it on the CPU.

### Predicting missing words

In [9]:
# Predict a missing words
results = fill_mask(
    "The capital of [MASK] is Rome.",
    top_k=5
)

for result in results:
    print(
        f"{result['token_str']}: "
        f"{result['score']:.3f}"
    )

italy: 0.699
rome: 0.043
india: 0.036
greece: 0.026
africa: 0.026


[MASK] marks the missing token. The model returns five candidates and their scores. The surrounding words including those after the mask help determine the predictions. This examples highlight that not all predicted words are plausible.

In [10]:
# Change the context
fill_mask("She deposited money at the [MASK].", top_k=5)

[{'score': 0.23860383033752441,
  'token': 2924,
  'token_str': 'bank',
  'sequence': 'she deposited money at the bank.'},
 {'score': 0.1868678480386734,
  'token': 2795,
  'token_str': 'table',
  'sequence': 'she deposited money at the table.'},
 {'score': 0.11572664231061935,
  'token': 4624,
  'token_str': 'desk',
  'sequence': 'she deposited money at the desk.'},
 {'score': 0.09614014625549316,
  'token': 4675,
  'token_str': 'counter',
  'sequence': 'she deposited money at the counter.'},
 {'score': 0.07184445112943649,
  'token': 2341,
  'token_str': 'door',
  'sequence': 'she deposited money at the door.'}]

In [11]:
fill_mask("She sat beside the river on the [MASK].", top_k=5)

[{'score': 0.14487625658512115,
  'token': 3509,
  'token_str': 'beach',
  'sequence': 'she sat beside the river on the beach.'},
 {'score': 0.12901127338409424,
  'token': 2924,
  'token_str': 'bank',
  'sequence': 'she sat beside the river on the bank.'},
 {'score': 0.04927145317196846,
  'token': 2940,
  'token_str': 'hill',
  'sequence': 'she sat beside the river on the hill.'},
 {'score': 0.040424324572086334,
  'token': 5749,
  'token_str': 'rocks',
  'sequence': 'she sat beside the river on the rocks.'},
 {'score': 0.03375193104147911,
  'token': 7424,
  'token_str': 'porch',
  'sequence': 'she sat beside the river on the porch.'}]

The word 'bank' has a different predictions (0.1868 vs 0.0492) as it depends on the context.  There are more sevaral answers possible, but these may not alwasy be plausible. BERT ranks the tokens based on the context, and how the words fits it. However, its scores do not establish factual truth. 

### Changing the context

In [12]:
# Test how changing the context affects BERT’s predictions.
sentences = [
    "She deposited money at the [MASK].",
    "She sat beside the river on the [MASK]."
]

for sentence in sentences:
    print(f"\nSentence: {sentence}")

    for result in fill_mask(sentence, top_k=5):
        print(f"{result['token_str']}: {result['score']:.3f}")


Sentence: She deposited money at the [MASK].
bank: 0.239
table: 0.187
desk: 0.116
counter: 0.096
door: 0.072

Sentence: She sat beside the river on the [MASK].
beach: 0.145
bank: 0.129
hill: 0.049
rocks: 0.040
porch: 0.034


The same models has been used for both sentences, only the input changes. 
'Bank' appear in both lists but the ranks and the scores are higher (0.239 vs 0.129) then in the example above. The surrounding context supports different meaning: a financial institution vs riverbank. BERT also suggest different alternatives that fit the context. 

Context changes both the suggested words and their scores. An open-ended sentence may have several plausible completions, so the highest-scoring prediction is not necessarily the only valid answer.

In [13]:
#Context change after missing word

sentences = [
    "The capital of [MASK] is Rome.",
    "The capital of [MASK] is Paris.",
    "The capital of [MASK] is Tokyo."
]

for sentence in sentences:
    print(f"\n{sentence}")

    for result in fill_mask(sentence, top_k=3):
        print(f"{result['token_str']}: {result['score']:.3f}")


The capital of [MASK] is Rome.
italy: 0.699
rome: 0.043
india: 0.036

The capital of [MASK] is Paris.
france: 0.928
brittany: 0.008
algeria: 0.007

The capital of [MASK] is Tokyo.
japan: 0.924
prefecture: 0.048
tokyo: 0.015


BERT uses context on both sides of [MASK]. The city appearing after the mask helps it predict the country.

**Takeway:** Changing the city after the masked token changed BERT’s predicted country, illustrating its use of surrounding context. All three top predictions were correct, but their confidence scores differed per sentence. This reinforced that prediction scores and factual correctness are distinct, and that a few successful examples do not establish overall model reliability.

## 2. GPT-2 - text-completion model

In [14]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="openai-community/gpt2",
    device=-1
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [15]:
result = generator(
    "Learning a new language is useful because",
    max_new_tokens=40,
    do_sample=False,  # chooses the highest-scoring next token
    pad_token_id=generator.tokenizer.eos_token_id
)

print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Learning a new language is useful because it allows you to learn new things.

The language is not a language. It is a language.

The language is not a language. It is a language.

The language


This output shows repetition, there is no coherent meaning. The model produces grammatical sentences, but is contradicting and repetitive. With do_sample=False, it repeatedly chooses the highest-scoring next token, which contributes to repetitive loops, although the model and prompt also influence the result.

#### Using the same prompt but changing how tokens are selected

In [16]:
# Import model 
from transformers import set_seed

set_seed(42)

result = generator(
    "Learning a new language is useful because",
    max_new_tokens=40,
    do_sample=True,         # samples from possible next tokens, sampling may improve variety, but does not guarantee correctness.
    temperature=0.7,        # favours more likely options while allowing variation.
    top_p=0.9,              # limits sampling to a set of likely candidates.
    pad_token_id=generator.tokenizer.eos_token_id
)

print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'do_sample', 'max_new_tokens', 'temperature', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Learning a new language is useful because it gives us a chance to learn new things, to learn new things and to learn new things again. The process of learning new things is different for everyone. You're not going to learn new things


**The generation settings are changed:** This continuation is more relevant, but still repetitive. It repeatedly uses 'learn new things'. Sampling changed the output without fully solving the problem.

#### Using the same settings and adding one repetition constraint

In [17]:
set_seed(42)

result = generator(
    "Learning a new language is useful because",
    max_new_tokens=40,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    no_repeat_ngram_size=3,        # prevents the same sequence of three tokens from appearing again.
    pad_token_id=generator.tokenizer.eos_token_id
)

print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'no_repeat_ngram_size', 'pad_token_id', 'do_sample', 'max_new_tokens', 'temperature', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Learning a new language is useful because it gives us a chance to learn new things, to learn to see what's new, to see how things change.

But what happens when we don't learn?

The answer is


This version is less repetitive and more varied, but it remains vague and shifts into a question rather than developing the original point. The unfinished ending likely reflects the 40-token limit.

**Key takeway**: Generation settings influence the output, but do not guarantee quality. 

* Greedy generation: Repetition and contradiction
* Sampling: More relevant text, but repeated phrases
* Sampling with a repetition constrain: Less repetition, but still limited substance

Using GPT-2, I compared how generation settings affected text completion. Sampling produced a more relevant continuation than greedy generation, while a repetition constraint reduced repeated phrases. However, the output remained vague. **This showed that generation settings influence fluency and variety without guaranteeing meaningful content.** These changes affected token selection during generation.

## 3. T5 (FLAN-T5-small)

In [18]:
# Import model 
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(model_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
t5_model.eval()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [19]:
import torch

text = (
    "A local library extended its opening hours during exam season. "
    "It now closes at 10 pm instead of 6 pm on weekdays. "
    "The change gives students more time to study. "
    "Weekend opening hours remain unchanged."
)

inputs = t5_tokenizer(
    "Summarize in one sentence: " + text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = t5_model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

summary = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(summary)

The library has extended its opening hours for the weekdays.


The encoder processes the input text and the decoder generates the summary.

The summary is broadly accurate but incomplete. 
- Captured: the library extended its weekday opening hours.
- Omitted: closing changed from 6 pm to 10 pm, the change applies during exam season, and weekends remain unchanged.

In [20]:
# More speciifc instructions
inputs = t5_tokenizer(
    "Summarize in one sentence. Include the new closing time "
    "and when the change applies: " + text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = t5_model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

print(t5_tokenizer.decode(outputs[0], skip_special_tokens=True))

Open the library at 10 pm on weekdays.


This output is incorrect. The expected words 'library' and '10 pm' did not prevent the model from confusing opening and closing times. The source says the library closes at 10 pm, but the model says to open it at 10 pm. It changes the statement into an instruction. 
Human Review is essential.

**Takeway**: In the T5 summarisation task, the first output broadly captured the main change but omitted important details. A more specific prompt produced a factual error, changing the library’s closing time into an opening time. it included the time, but changed its meaning. 

**Learnings:**

Through practical experiments with **BERT, GPT-2 and FLAN-T5**, I explored how Transformer architectures support different NLP tasks. 
* BERT demonstrated how context influences masked-token predictions and scores and ranks. 
* GPT-2 showed that generation settings can reduce repetition without guaranteeing meaningful content. 
* FLAN-T5 produced both an accurate but incomplete summary and a factual error, highlighting the need to check outputs against their source. 


The results showed tha a more specific prompt does not guarantee a better answer and can perform even worse as per example and it does not gurantee a faithful summarised output. Evaluation must always check the meaning against the information source. Confident sounding and fluent output, instructions following or even a presence of expected words alone does not gurantee the output correctness. Human review is critical.
